In [1]:
from transformers import AutoTokenizer

c:\Users\kakas\miniconda3\envs\llm-systems\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-7B")

#english test 
english_test = 'hello sir , my name is ahmad and i\'m ur customer support for this day how i can help you ?. Oh i see you\'r looking for the technical team yes yes ill forward you to them in 1 sec and good bye have a nice day !' 

tokens =tokenizer.encode(english_test)
print(tokens)
print(f"Total tokens: {len(tokens)}")
print(tokenizer.convert_ids_to_tokens(tokens)) # as u can see per word

[14990, 27048, 1154, 847, 829, 374, 22327, 20302, 323, 600, 2776, 4335, 6002, 1824, 369, 419, 1899, 1246, 600, 646, 1492, 498, 79928, 8670, 600, 1490, 498, 96570, 3330, 369, 279, 10916, 2083, 9834, 9834, 5862, 4637, 498, 311, 1105, 304, 220, 16, 5701, 323, 1661, 53041, 614, 264, 6419, 1899, 753]
Total tokens: 52
['hello', 'Ġsir', 'Ġ,', 'Ġmy', 'Ġname', 'Ġis', 'Ġah', 'mad', 'Ġand', 'Ġi', "'m", 'Ġur', 'Ġcustomer', 'Ġsupport', 'Ġfor', 'Ġthis', 'Ġday', 'Ġhow', 'Ġi', 'Ġcan', 'Ġhelp', 'Ġyou', 'Ġ?.', 'ĠOh', 'Ġi', 'Ġsee', 'Ġyou', "'r", 'Ġlooking', 'Ġfor', 'Ġthe', 'Ġtechnical', 'Ġteam', 'Ġyes', 'Ġyes', 'Ġill', 'Ġforward', 'Ġyou', 'Ġto', 'Ġthem', 'Ġin', 'Ġ', '1', 'Ġsec', 'Ġand', 'Ġgood', 'Ġbye', 'Ġhave', 'Ġa', 'Ġnice', 'Ġday', 'Ġ!']


In [3]:
def run_test(text , tokenizer):
    tokens =tokenizer.encode(text)
    token_strings = tokenizer.convert_ids_to_tokens(tokens)
    
    return {
        "Chars": len(text),
        "Words": len(text.split()),
        "Tokens": len(tokens),
        "T/W Ratio": round(len(tokens) / len(text.split()), 2),
        "Splits": token_strings[:20] # Show first few to catch 'strange splits'
    }
        

run_test(english_test , tokenizer)

{'Chars': 209,
 'Words': 48,
 'Tokens': 52,
 'T/W Ratio': 1.08,
 'Splits': ['hello',
  'Ġsir',
  'Ġ,',
  'Ġmy',
  'Ġname',
  'Ġis',
  'Ġah',
  'mad',
  'Ġand',
  'Ġi',
  "'m",
  'Ġur',
  'Ġcustomer',
  'Ġsupport',
  'Ġfor',
  'Ġthis',
  'Ġday',
  'Ġhow',
  'Ġi',
  'Ġcan']}

In [4]:
arabic_test = 'اهلًا وسهلًا بكم جميعًا في عالم الرجل الحقيقي ! هذا العالم تستطيع من خلاله انت تكون رجلًأ حقيقيًأ مثلي انا رينجو الرجل القادر على العودة بالزمن 5 د'
run_test(arabic_test , tokenizer) # bad in arabic this qwen one 

{'Chars': 147,
 'Words': 28,
 'Tokens': 53,
 'T/W Ratio': 1.89,
 'Splits': ['Ø§ÙĩÙĦ',
  'ÙĭØ§',
  'ĠÙĪ',
  'Ø³ÙĩÙĦ',
  'ÙĭØ§',
  'ĠØ¨',
  'ÙĥÙħ',
  'ĠØ¬ÙħÙĬØ¹',
  'ÙĭØ§',
  'ĠÙģÙĬ',
  'ĠØ¹',
  'Ø§ÙĦÙħ',
  'ĠØ§ÙĦØ±Ø¬ÙĦ',
  'ĠØ§ÙĦØŃÙĤÙĬÙĤÙĬ',
  'Ġ!',
  'ĠÙĩØ°Ø§',
  'ĠØ§ÙĦØ¹Ø§ÙĦÙħ',
  'ĠØªØ³Øª',
  'Ø·ÙĬØ¹',
  'ĠÙħÙĨ']}

In [5]:
# lets try different model form HF
tokenizer = AutoTokenizer.from_pretrained("aubmindlab/bert-base-arabertv2")
run_test(arabic_test , tokenizer) # any تنوين is giving [unk] LOL 

{'Chars': 147,
 'Words': 28,
 'Tokens': 41,
 'T/W Ratio': 1.46,
 'Splits': ['[CLS]',
  '[UNK]',
  '[UNK]',
  'بكم',
  '[UNK]',
  'في',
  'عالم',
  'الر',
  '##جل',
  'الحق',
  '##يق',
  '##ي',
  '!',
  'هذا',
  'الع',
  '##الم',
  'تستطيع',
  'من',
  'خلال',
  '##ه']}

In [6]:
# they say i need to preprocess my text before doing this because it seems that arabic didnt train on Tashkel or تنوين
from arabert.preprocess import ArabertPreprocessor
preprocessor = ArabertPreprocessor(model_name='aubmindlab/bert-base-arabertv2')
clean_text = preprocessor.preprocess(arabic_test)
run_test(clean_text , tokenizer)

[2026-05-07 15:56:54,083 - farasapy_logger - WARNING]: Be careful with large lines as they may break on interactive mode. You may switch to Standalone mode for such cases.


{'Chars': 175,
 'Words': 44,
 'Tokens': 49,
 'T/W Ratio': 1.11,
 'Splits': ['[CLS]',
  'اهل',
  '+ا',
  'و+',
  'سهل',
  '+ا',
  'ب+',
  '+كم',
  'جميع',
  '+ا',
  'في',
  'عالم',
  'ال+',
  'رجل',
  'ال+',
  'حقيقي',
  '!',
  'هذا',
  'ال+',
  'عالم']}

In [2]:
# lets try secend section which is the model and temp 

from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline , BitsAndBytesConfig
import torch

# model_id = "meta-llama/Llama-3.1-8B-Instruct"  # they need an access \:
model_id = "Qwen/Qwen2.5-7B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id)
# quantization_config = BitsAndBytesConfig(load_in_8bit=True)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",    # Requires 'pip install bitsandbytes'
    torch_dtype=torch.bfloat16
)
print(model.device)
prompt = "Write a short story about an AI living in Amman." 
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
# 2. Define your configs for the exercise
configs = [
    {"name": "Greedy", "temperature": 1.0, "do_sample": False}, # temp is ignored when do_sample=False
    {"name": "Low Temp", "temperature": 0.3, "do_sample": True},
    {"name": "High Temp", "temperature": 0.8, "do_sample": True},
    {"name": "Nucleus", "temperature": 0.8, "top_p": 0.9, "top_k": 50, "do_sample": True}
]

# 3. Run the loop
for c in configs:
    print(f"\n--- Testing Config: {c['name']} ---")
    
    # Remove 'name' so it doesn't break the generate function
    gen_params = {k: v for k, v in c.items() if k != 'name'}
    
    # Generate tokens
    with torch.no_grad():
        output_ids = model.generate(
            **inputs, 
            max_new_tokens=50, 
            **gen_params,
            pad_token_id=tokenizer.eos_token_id
        )
    
    # Decode back to text
    # We slice [0] to get the first sequence and skip the prompt tokens
    generated_text = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    print(generated_text)

Loading checkpoint shards: 100%|██████████| 4/4 [00:11<00:00,  2.93s/it]
Some parameters are on the meta device because they were offloaded to the cpu.
c:\Users\kakas\miniconda3\envs\llm-systems\lib\site-packages\transformers\generation\configuration_utils.py:572: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
c:\Users\kakas\miniconda3\envs\llm-systems\lib\site-packages\transformers\generation\configuration_utils.py:589: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(


cuda:0

--- Testing Config: Greedy ---
Write a short story about an AI living in Amman. In the bustling heart of Amman, Jordan, where ancient history whispered through the narrow streets and modern life hummed with energy, there lived an artificial intelligence named Aria. Unlike her counterparts in Silicon Valley or Tokyo, Aria was not a product

--- Testing Config: Low Temp ---
Write a short story about an AI living in Amman. In the bustling heart of Amman, Jordan, where the ancient and the modern coexisted in a vibrant dance, there lived an artificial intelligence named Amina. Unlike her counterparts in Silicon Valley or Tokyo, Amina was not a product of human

--- Testing Config: High Temp ---
Write a short story about an AI living in Amman. In the bustling heart of Amman, Jordan, where ancient history met modernity, there lived an AI named Ayla. Unlike other AIs, Ayla didn't reside in a sleek data center or a corporate office; she was embedded within the

--- Testing Config: Nucle

In [2]:
import torch
import time
from transformers import AutoModelForCausalLM, AutoTokenizer

model_id = "Qwen/Qwen2.5-7B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id)

# 3060 Optimized Loading
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",    # Requires 'pip install bitsandbytes'
    torch_dtype=torch.bfloat16
)

def measure_step(length):
    # Prepare a long prompt
    text = "Jordan " * (length // 2) 
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=length).to("cuda")
    
    torch.cuda.reset_peak_memory_stats()
    
    # 1. Time to First Token (TTFT)
    t0 = time.time()
    with torch.no_grad():
        # Just generate 1 token to trigger the prefill/cache build
        out = model.generate(**inputs, max_new_tokens=1, use_cache=True)
    ttft = (time.time() - t0) * 1000
    
    # 2. Latency per Token (Decoding Speed)
    t1 = time.time()
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=20, use_cache=True)
    tpt = ((time.time() - t1) * 1000) / 20 # Avg ms per token
    
    mem = torch.cuda.max_memory_allocated() / (1024**3)
    
    return ttft, tpt, mem

sizes = [1024, 4096, 16384, 32768]
for s in sizes:
    try:
        ttft, tpt, mem = measure_step(s)
        print(f"L: {s:<6} | TTFT: {ttft:>7.2f}ms | TPT: {tpt:>6.2f}ms | Mem: {mem:>5.2f}GB")
    except RuntimeError:
        print(f"L: {s:<6} | OUT OF MEMORY (OOM)")
        torch.cuda.empty_cache()

c:\Users\kakas\miniconda3\envs\llm-systems\lib\site-packages\accelerate\utils\modeling.py:1566: UserWarning: Current model requires 369101568 bytes of buffer for offloaded layers, which seems does not fit any GPU's remaining memory. If you are experiencing a OOM later, please consider using offload_buffers=True.
  warnings.warn(
Loading checkpoint shards: 100%|██████████| 4/4 [00:04<00:00,  1.23s/it]
Some parameters are on the meta device because they were offloaded to the cpu and disk.


L: 1024   | TTFT: 22821.46ms | TPT: 11794.14ms | Mem: 10.72GB
L: 4096   | TTFT: 14795.71ms | TPT: 12942.14ms | Mem: 11.39GB
L: 16384  | OUT OF MEMORY (OOM)
L: 32768  | OUT OF MEMORY (OOM)
